# 📋 Sprint 4 — Report Generation, Nutrition Advice & Delivery
**Project:** Autonomous Medical Research & Patient Triage Assistant
**Day:** 4 of 5 | **Sprint Goal:** Generate structured triage reports + deliver via Telegram/WhatsApp

## What we build today
```
Triage Classification (Sprint 3)
  -> Patient Report  : Plain language + urgency + action
  -> Clinician Report: Differential + ICD-10 + drugs + PubMed refs
  -> Self-Care Module: Nigerian nutrition advice + home remedies
  -> Progress Monitor: 24-hour follow-up scheduler
  -> Delivery        : Telegram + WhatsApp formatted messages
```

## Sprint 4 User Stories
> **US-04:** Clinician receives differential diagnosis with ICD-10 codes and PubMed citations
> **US-05:** Patient on self-care receives Nigerian food nutrition advice
> **US-06:** Patient receives 24-hour follow-up progress check
> **US-07:** Every response includes safety disclaimer

**Definition of Done:**
- Both patient and clinician reports generated correctly
- Nigerian nutrition advice included for self-care cases
- 24-hour follow-up scheduled for self-care cases
- Safety disclaimer on 100% of responses

In [ ]:
import os, json, time, uuid, pathlib
from datetime import datetime, timezone
from typing import TypedDict, Optional, List, Dict
from dotenv import load_dotenv
from openai import OpenAI
from langgraph.graph import StateGraph, END

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
assert OPENAI_API_KEY, 'Set OPENAI_API_KEY'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
REPORT_MODEL  = 'gpt-4o'
FAST_MODEL    = 'gpt-4o-mini'

TELEGRAM_BOT_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN', '')
TELEGRAM_CHAT_ID   = os.getenv('TELEGRAM_CHAT_ID', '')

print('Clients initialised')
print(f'Report model    : {REPORT_MODEL}')
print(f'Telegram config : {"set" if TELEGRAM_BOT_TOKEN else "not set (optional for notebook)"}')

In [ ]:
class MedicalAgentState(TypedDict):
    session_id:        str
    ticket_id:         str
    initiated_at:      str
    channel:           str
    chat_id:           str
    user_mode:         str
    raw_message:       str
    symptoms:          List[str]
    duration:          Optional[str]
    age:               Optional[str]
    medications:       List[str]
    status:            str
    errors:            List[str]
    raw_research:      Optional[str]
    research_chunks:   Optional[List[str]]
    pinecone_ids:      Optional[List[str]]
    retrieved_chunks:  Optional[List[str]]
    reranked_chunks:   Optional[List[str]]
    urgency_level:     Optional[str]
    urgency_score:     Optional[int]
    differential:      Optional[List[str]]
    red_flags:         Optional[List[str]]
    retry_count:       int
    triage_report:     Optional[Dict[str, str]]
    nutrition_advice:  Optional[str]
    home_remedies:     Optional[str]
    follow_up_sent:    bool
    report_ready:      bool
    workflow_path:     List[str]

print('MedicalAgentState schema ready')

In [ ]:
if pathlib.Path('sprint3_medical_output.json').exists():
    data = json.loads(pathlib.Path('sprint3_medical_output.json').read_text())
    sprint3_state = MedicalAgentState(**{k: data.get(k) for k in MedicalAgentState.__annotations__})
    print(f'Loaded Sprint 3 state')
    print(f'  Symptoms       : {sprint3_state["symptoms"][:3]}')
    print(f'  Urgency        : {sprint3_state.get("urgency_level")}')
    print(f'  Differential   : {sprint3_state.get("differential", [])[:2]}')
else:
    # Test state if Sprint 3 not available
    sprint3_state = MedicalAgentState(
        session_id=str(uuid.uuid4()),
        ticket_id=f'MED-{datetime.now(timezone.utc).strftime("%Y%m%d")}-TEST0001',
        initiated_at=datetime.now(timezone.utc).isoformat(),
        channel='telegram', chat_id='8138298582', user_mode='patient',
        raw_message='I have malaria fever chills headache for 3 days',
        symptoms=['malaria', 'fever', 'chills', 'headache'],
        duration='3 days', age=None, medications=[],
        status='triaged', errors=[],
        raw_research=None, research_chunks=None, pinecone_ids=None,
        retrieved_chunks=None,
        reranked_chunks=[
            'Malaria is a life-threatening disease caused by Plasmodium parasites. In Nigeria, Plasmodium falciparum is the dominant species. WHO recommends Artemisinin-based Combination Therapies (ACTs) as first-line treatment.',
            'NHIS Nigeria covers malaria treatment. Recommended tests include RDT and blood smear. Signs requiring urgent care: high fever >39C, vomiting, inability to take oral medication.',
            'Self-care for mild malaria: rest, hydration, paracetamol for fever. Nutrition: bitter leaf soup, plenty of fluids, citrus fruits for vitamin C.'
        ],
        urgency_level='urgent', urgency_score=7,
        differential=['Malaria (P. falciparum)', 'Typhoid fever', 'Viral fever'],
        red_flags=[], retry_count=0,
        triage_report={'_classification': json.dumps({
            'urgency_level': 'urgent', 'urgency_score': 7,
            'differential': ['Malaria (P. falciparum)', 'Typhoid fever', 'Viral fever'],
            'recommended_action': 'Visit clinic or hospital for RDT test and treatment within 24 hours',
            'recommended_tests': ['Rapid Diagnostic Test (RDT)', 'Blood smear', 'Full Blood Count'],
            'red_flags_present': [],
            'self_care_eligible': False,
            'facility_needed': 'clinic',
            'confidence': 'high'
        })},
        nutrition_advice=None, home_remedies=None,
        follow_up_sent=False, report_ready=False, workflow_path=['retrieve', 'rerank', 'triage_agent']
    )
    print('No Sprint 3 output found - using test state with malaria case')

In [ ]:
# Nigerian food recommendations by condition
NIGERIAN_NUTRITION = {
    'malaria': {
        'eat':   ['Bitter leaf soup', 'Citrus fruits (orange, lemon)', 'Plenty water and fluids', 'Light pap (akamu)', 'Boiled yam'],
        'avoid_during': ['Heavy oily foods', 'Alcohol', 'Spicy pepper soup (until recovering)'],
        'tip':   'Drink at least 2-3 litres of water daily. Vitamin C helps immunity.'
    },
    'typhoid': {
        'eat':   ['Light pap (akamu)', 'Oats', 'Boiled yam or plantain', 'Plenty clean water', 'Fruit juice without sugar'],
        'avoid_during': ['Pepper', 'Fried or oily food', 'Street food', 'Raw vegetables'],
        'tip':   'Eat small portions frequently. Wash hands before eating.'
    },
    'anaemia': {
        'eat':   ['Ugu (pumpkin leaves)', 'Liver (chicken or cow)', 'Beans and lentils', 'Ofada rice', 'Watermelon', 'Dates'],
        'avoid_during': ['Tea or coffee with meals (blocks iron)'],
        'avoid_long_term': ['Alcohol if it is regular/excessive'],
        'tip':   'Eat vitamin C (orange, lemon) with iron-rich foods to help absorption.'
    },
    'hypertension': {
        'eat':   ['Garden egg', 'Fish (grilled)', 'Oats', 'Vegetables', 'Banana', 'Watermelon'],
        'avoid_long_term': ['Salt and seasoning cubes', 'Fried foods', 'Red meat', 'Alcohol', 'Sugary drinks'],
        'tip':   'Cook without Maggi or salt. Use herbs and spices for flavour.'
    },
    'diabetes': {
        'eat':   ['Tiger nuts', 'Unripe plantain', 'Garden egg', 'Leafy vegetables', 'Fish', 'Brown rice'],
        'avoid_long_term': ['Eba/fufu (large portions)', 'White bread', 'Sugary drinks', 'Puff puff', 'Sweet chin chin'],
        'tip':   'Eat at regular times. Check blood sugar before and after meals.'
    },
    'fever': {
        'eat':   ['Plenty water and fluids', 'Fruit juice', 'Light pap', 'Coconut water', 'Vegetable soup'],
        'avoid_during': ['Heavy meals', 'Alcohol', 'Very spicy food'],
        'tip':   'Stay hydrated. Rest. Take paracetamol for fever above 38.5C.'
    },
    'default': {
        'eat':   ['Plenty water', 'Fruits and vegetables', 'Light easily digestible foods', 'Local soups with vegetables'],
        'avoid_during': ['Alcohol', 'Very oily or spicy food during illness'],
        'tip':   'Rest well and stay hydrated. Eat small portions regularly.'
    }
}

def get_nutrition_advice(symptoms: List[str], differential: List[str]) -> str:
    all_text = ' '.join(symptoms + differential).lower()
    for condition in NIGERIAN_NUTRITION:
        if condition in all_text:
            data = NIGERIAN_NUTRITION[condition]
            parts = [f'FOODS TO EAT:\n' + '\n'.join([f'  - {f}' for f in data['eat']])]
            if data.get('avoid_during'):
                parts.append('FOODS TO AVOID DURING ILLNESS:\n' + '\n'.join([f'  - {f}' for f in data['avoid_during']]))
            if data.get('avoid_long_term'):
                parts.append('FOODS TO AVOID LONG-TERM:\n' + '\n'.join([f'  - {f}' for f in data['avoid_long_term']]))
            parts.append(f'TIP: {data["tip"]}')
            return '\n\n'.join(parts)
    data = NIGERIAN_NUTRITION['default']
    parts = [f'GENERAL GUIDANCE:\n' + '\n'.join([f'  - {f}' for f in data['eat']])]
    if data.get('avoid_during'):
        parts.append('FOODS TO AVOID DURING ILLNESS:\n' + '\n'.join([f'  - {f}' for f in data['avoid_during']]))
    if data.get('avoid_long_term'):
        parts.append('FOODS TO AVOID LONG-TERM:\n' + '\n'.join([f'  - {f}' for f in data['avoid_long_term']]))
    parts.append(f'TIP: {data["tip"]}')
    return '\n\n'.join(parts)

# Test
print('Nutrition advice test for malaria:')
print(get_nutrition_advice(['malaria', 'fever'], ['Malaria (P. falciparum)']))

In [ ]:
URGENCY_EMOJIS = {
    'emergency': '🔴',
    'urgent':    '🟠',
    'standard':  '🟡',
    'self-care': '🟢'
}

URGENCY_ACTIONS = {
    'emergency': 'Go to A&E IMMEDIATELY. Call 112 now.',
    'urgent':    'See a doctor within 24 hours.',
    'standard':  'Book a clinic appointment this week.',
    'self-care': 'You can manage this at home with the guidance below.'
}

def generate_patient_report_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Generate plain-language patient triage report.
    Includes urgency, action, possible conditions, nutrition if self-care.
    """
    urgency    = state.get('urgency_level', 'standard')
    score      = state.get('urgency_score', 5)
    diff       = state.get('differential', [])
    symptoms   = state.get('symptoms', [])
    ticket     = state.get('ticket_id', '')
    errors     = list(state.get('errors', []))
    chunks     = state.get('reranked_chunks') or state.get('retrieved_chunks') or []
    context    = '\n\n'.join(chunks[:3])

    print(f'[REPORT] Generating patient report | Urgency: {urgency.upper()}')

    emoji  = URGENCY_EMOJIS.get(urgency, '🟡')
    action = URGENCY_ACTIONS.get(urgency, 'See a doctor.')

    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=REPORT_MODEL, temperature=0.2,
                messages=[
                    {'role': 'system', 'content': (
                        'You are a patient health assistant in Nigeria. '
                        'Write a clear, simple triage report for a patient. '
                        'Use plain language. No medical jargon. '
                        'Be compassionate but direct. '
                        'Base information on the provided clinical context.'
                    )},
                    {'role': 'user', 'content': (
                        f'Clinical context:\n{context}\n\n'
                        f'Urgency: {urgency} ({score}/10)\n'
                        f'Likely conditions: {diff}\n'
                        f'Symptoms: {symptoms}\n\n'
                        f'Write a 150-word patient report covering:\n'
                        f'1. What might be causing symptoms (in simple terms)\n'
                        f'2. What tests to get if seeing a doctor\n'
                        f'3. Warning signs to watch for\n'
                        f'4. General care advice\n'
                        f'Do NOT include the disclaimer - it will be added separately.'
                    )}
                ]
            )
            report_body = resp.choices[0].message.content.strip()

            # Build full formatted report
            full_report = (
                f'{emoji} TRIAGE ASSESSMENT\n'
                f'Reference: {ticket}\n'
                f'Urgency Level: {urgency.upper()} ({score}/10)\n\n'
                f'WHAT TO DO: {action}\n\n'
                f'{report_body}\n\n'
                f'DISCLAIMER: This assessment is for guidance only and is NOT a substitute '
                f'for professional medical advice. Always consult a qualified healthcare '
                f'provider for diagnosis and treatment.'
            )

            existing = state.get('triage_report') or {}
            existing['patient_report'] = full_report

            print(f'[REPORT] Patient report generated ({len(full_report)} chars)')

            return {
                **state, 'triage_report': existing,
                'status': 'report_generated', 'errors': errors,
                'workflow_path': state.get('workflow_path', []) + ['patient_report']
            }
        except Exception as e:
            if attempt == 2:
                return {
                    **state, 'status': 'error_report_failed',
                    'errors': errors + [f'Patient report failed: {e}'],
                    'workflow_path': state.get('workflow_path', []) + ['patient_report']
                }
            time.sleep(2 ** attempt)

print('generate_patient_report_node defined')

In [ ]:
def generate_clinician_report_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Generate clinical report for healthcare providers.
    Includes differential diagnosis with ICD-10, drug recommendations, PubMed refs.
    """
    urgency  = state.get('urgency_level', 'standard')
    score    = state.get('urgency_score', 5)
    diff     = state.get('differential', [])
    symptoms = state.get('symptoms', [])
    meds     = state.get('medications', [])
    ticket   = state.get('ticket_id', '')
    errors   = list(state.get('errors', []))
    chunks   = state.get('reranked_chunks') or state.get('retrieved_chunks') or []
    context  = '\n\n'.join(chunks[:3])

    # Get drug interaction data from Sprint 3
    report_data = state.get('triage_report') or {}
    drug_data   = {}
    if 'drug_interactions' in report_data:
        try:
            drug_data = json.loads(report_data['drug_interactions'])
        except Exception:
            pass

    print(f'[CLINICIAN REPORT] Generating clinical report | Urgency: {urgency.upper()}')

    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=REPORT_MODEL, temperature=0.1,
                messages=[
                    {'role': 'system', 'content': (
                        'You are a clinical decision support system for Nigerian healthcare providers. '
                        'Generate a structured clinical report using the provided evidence context. '
                        'Include ICD-10 codes, NHIS protocols, and cite any PubMed evidence. '
                        'Be specific, clinical, and concise.'
                    )},
                    {'role': 'user', 'content': (
                        f'Clinical evidence context:\n{context}\n\n'
                        f'Presentation: {symptoms}\n'
                        f'Current medications: {meds or "None"}\n'
                        f'Drug interactions noted: {drug_data.get("interactions", [])}\n\n'
                        f'Generate a clinical report with:\n'
                        f'1. Top 3 differential diagnoses with ICD-10 codes\n'
                        f'2. Recommended investigations (with rationale)\n'
                        f'3. First-line treatment options (drugs, doses, routes)\n'
                        f'4. NHIS Nigeria protocol reference if applicable\n'
                        f'5. PubMed citation from context if available\n'
                        f'6. Referral criteria\n'
                        f'Keep under 300 words. Clinical shorthand acceptable.'
                    )}
                ]
            )
            report_body = resp.choices[0].message.content.strip()

            full_report = (
                f'CLINICAL TRIAGE REPORT\n'
                f'Reference: {ticket}\n'
                f'Urgency: {urgency.upper()} ({score}/10)\n'
                f'Generated: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}\n\n'
                f'{report_body}\n\n'
                f'DISCLAIMER: This is a clinical decision support tool. '
                f'Clinical judgment of the treating physician supersedes this report.'
            )

            existing = state.get('triage_report') or {}
            existing['clinician_report'] = full_report

            print(f'[CLINICIAN REPORT] Generated ({len(full_report)} chars)')

            return {
                **state, 'triage_report': existing,
                'status': 'report_generated', 'errors': errors,
                'workflow_path': state.get('workflow_path', []) + ['clinician_report']
            }
        except Exception as e:
            if attempt == 2:
                return {
                    **state, 'status': 'error_report_failed',
                    'errors': errors + [f'Clinician report failed: {e}'],
                    'workflow_path': state.get('workflow_path', []) + ['clinician_report']
                }
            time.sleep(2 ** attempt)

print('generate_clinician_report_node defined')

In [ ]:
def self_care_nutrition_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Add Nigerian nutrition advice and home remedies for self-care cases.
    Only runs when urgency_level == 'self-care'.
    """
    urgency  = state.get('urgency_level', 'standard')
    symptoms = state.get('symptoms', [])
    diff     = state.get('differential', [])
    errors   = list(state.get('errors', []))

    if urgency != 'self-care':
        print(f'[SELF-CARE] Urgency is {urgency} - skipping nutrition module')
        return {
            **state,
            'workflow_path': state.get('workflow_path', []) + ['self_care_skipped']
        }

    print('[SELF-CARE] Generating Nigerian nutrition advice...')

    nutrition = get_nutrition_advice(symptoms, diff)

    # Generate home remedies via OpenAI
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=FAST_MODEL, temperature=0.2,
                messages=[
                    {'role': 'system', 'content': (
                        'You are a Nigerian community health advisor. '
                        'Suggest ONLY evidence-based, safe home remedies. '
                        'Never suggest unproven or potentially harmful traditional medicine. '
                        'Keep to 3-4 practical, safe, accessible suggestions.'
                    )},
                    {'role': 'user', 'content': (
                        f'Symptoms: {symptoms}\n'
                        f'Likely condition: {diff[0] if diff else "general illness"}\n'
                        f'Suggest safe home care measures available in Nigeria. '
                        f'Be brief and practical.'
                    )}
                ]
            )
            home_remedies = resp.choices[0].message.content.strip()

            print(f'[SELF-CARE] Nutrition advice and home remedies generated')

            return {
                **state,
                'nutrition_advice': nutrition,
                'home_remedies':    home_remedies,
                'errors':           errors,
                'workflow_path':    state.get('workflow_path', []) + ['self_care_nutrition']
            }
        except Exception as e:
            if attempt == 2:
                return {
                    **state,
                    'nutrition_advice': nutrition,
                    'errors': errors + [f'Home remedies generation failed: {e}'],
                    'workflow_path': state.get('workflow_path', []) + ['self_care_nutrition']
                }
            time.sleep(2 ** attempt)

print('self_care_nutrition_node defined')

In [ ]:
def deliver_report_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Format and deliver triage report via Telegram or WhatsApp.
    Selects correct report based on user_mode.
    Appends nutrition advice for self-care cases.
    """
    mode       = state.get('user_mode', 'patient')
    urgency    = state.get('urgency_level', 'standard')
    report     = state.get('triage_report') or {}
    nutrition  = state.get('nutrition_advice')
    remedies   = state.get('home_remedies')
    errors     = list(state.get('errors', []))

    # Select appropriate report
    if mode == 'clinician':
        message_body = report.get('clinician_report', report.get('patient_report', 'Report unavailable'))
    else:
        message_body = report.get('patient_report', report.get('emergency_response', 'Report unavailable'))

    # Append nutrition advice for self-care
    if urgency == 'self-care' and nutrition:
        message_body += f'\n\nNIGERIAN NUTRITION ADVICE:\n{nutrition}'

    if urgency == 'self-care' and remedies:
        message_body += f'\n\nHOME CARE GUIDANCE:\n{remedies}'

    # Add follow-up notice for self-care
    if urgency == 'self-care':
        message_body += (
            '\n\nPROGRESS MONITORING: You will receive a check-in message in 24 hours. '
            'Reply to let us know how you are feeling.'
        )

    # Print formatted message
    print(f'\n{"-" * 60}')
    print(f'TRIAGE REPORT DELIVERED ({mode.upper()} MODE)')
    print(f'{"-" * 60}')
    print(message_body[:800] + ('...' if len(message_body) > 800 else ''))
    print(f'{"-" * 60}')
    print(f'Total length: {len(message_body)} characters')

    # Send via Telegram if configured
    if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID:
        try:
            import urllib.request
            # Split long messages (Telegram limit: 4096 chars)
            chunks = [message_body[i:i+4000] for i in range(0, len(message_body), 4000)]
            for chunk in chunks:
                payload = json.dumps({'chat_id': TELEGRAM_CHAT_ID, 'text': chunk}).encode()
                req = urllib.request.Request(
                    f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage',
                    data=payload, headers={'Content-Type': 'application/json'}, method='POST'
                )
                urllib.request.urlopen(req)
                time.sleep(0.5)
            print(f'[DELIVER] Telegram delivery successful')
        except Exception as e:
            errors.append(f'Telegram delivery failed: {e}')
            print(f'[DELIVER] Telegram failed: {e}')
    else:
        print('[DELIVER] Telegram not configured - report printed above')

    return {
        **state, 'report_ready': True,
        'status': 'complete', 'errors': errors,
        'workflow_path': state.get('workflow_path', []) + ['deliver_report']
    }

print('deliver_report_node defined')

In [ ]:
def route(state):
    return 'error_handler' if 'error' in state.get('status', '') else 'continue'

def route_by_mode(state):
    return 'clinician_report' if state.get('user_mode') == 'clinician' else 'patient_report'

def error_handler_node(state):
    print('PIPELINE ERROR:')
    for e in state.get('errors', []): print(f'  {e}')
    return {**state, 'status': 'failed',
            'workflow_path': state.get('workflow_path', []) + ['error_handler']}

def build_sprint4_graph():
    g = StateGraph(MedicalAgentState)
    g.add_node('patient_report',   generate_patient_report_node)
    g.add_node('clinician_report', generate_clinician_report_node)
    g.add_node('self_care',        self_care_nutrition_node)
    g.add_node('deliver_report',   deliver_report_node)
    g.add_node('error_handler',    error_handler_node)

    # Route by mode: patient or clinician report
    g.set_entry_point('patient_report')
    g.add_conditional_edges('patient_report', route,
        {'continue': 'self_care', 'error_handler': 'error_handler'})
    g.add_edge('clinician_report', 'self_care')
    g.add_conditional_edges('self_care', route,
        {'continue': 'deliver_report', 'error_handler': 'error_handler'})
    g.add_conditional_edges('deliver_report', route,
        {'continue': END, 'error_handler': 'error_handler'})
    g.add_edge('error_handler', END)
    return g.compile()

sprint4_graph = build_sprint4_graph()
print('Sprint 4 graph compiled')
print('Flow: patient_report -> self_care -> deliver_report -> END')

In [ ]:
print('=' * 60)
print(f'Running Sprint 4 for: {sprint3_state["raw_message"][:50]}')
print(f'Urgency: {sprint3_state.get("urgency_level")}')
print(f'Mode   : {sprint3_state.get("user_mode")}')
print('=' * 60)

result = sprint4_graph.invoke(sprint3_state)

print(f'\nStatus        : {result["status"]}')
print(f'Report ready  : {result["report_ready"]}')
print(f'Workflow path : {" -> ".join(result["workflow_path"])}')
print(f'Errors        : {result.get("errors", [])}')

In [ ]:
print('=' * 60)
print('SPRINT 4 - DEFINITION OF DONE CHECK')
print('=' * 60)

report     = result.get('triage_report') or {}
mode       = result.get('user_mode', 'patient')
urgency    = result.get('urgency_level', 'standard')

has_report      = bool(report.get('patient_report') or report.get('emergency_response'))
has_disclaimer  = 'DISCLAIMER' in (report.get('patient_report', '') + report.get('clinician_report', '') + report.get('emergency_response', ''))
report_ready    = result.get('report_ready', False)
status_complete = result.get('status') == 'complete'

print(f'{"OK" if has_report     else "FAIL"} Report generated          : {"Yes" if has_report else "No"}')
print(f'{"OK" if has_disclaimer else "FAIL"} Safety disclaimer present : {"Yes" if has_disclaimer else "No"}')
print(f'{"OK" if report_ready   else "FAIL"} Report marked ready')
print(f'{"OK" if status_complete else "FAIL"} Status: {result.get("status")}')

if urgency == 'self-care':
    has_nutrition = bool(result.get('nutrition_advice'))
    print(f'{"OK" if has_nutrition else "FAIL"} Nigerian nutrition advice included')

all_ok = all([has_report, has_disclaimer, report_ready, status_complete])
print(f'\n{"Sprint 4 DoD: MET" if all_ok else "Sprint 4 DoD: PARTIAL"}')
print('Ready for Sprint 5 - QA, Documentation and Full Pipeline')

# Save for Sprint 5
pathlib.Path('sprint4_medical_output.json').write_text(
    json.dumps(dict(result), indent=2, default=str))
print('Sprint 4 output saved to sprint4_medical_output.json')

In [ ]:
# Quick test - verify report format for each urgency level
urgency_emojis = {'emergency': '🔴', 'urgent': '🟠', 'standard': '🟡', 'self-care': '🟢'}
actions = {
    'emergency': 'Go to A&E IMMEDIATELY. Call 112 now.',
    'urgent':    'See a doctor within 24 hours.',
    'standard':  'Book a clinic appointment this week.',
    'self-care': 'You can manage this at home with the guidance below.'
}

print('URGENCY LEVEL FORMAT TEST')
print('=' * 55)
for level, emoji in urgency_emojis.items():
    print(f'{emoji} {level.upper():12} -> {actions[level]}')

print('\nNIGERIAN NUTRITION TEST')
print('=' * 55)
test_conditions = ['malaria', 'typhoid', 'anaemia', 'hypertension']
for cond in test_conditions:
    advice = get_nutrition_advice([cond], [cond])
    first_food = advice.split('\n')[1].strip()
    print(f'{cond:15} -> {first_food}')